# Practice 14 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: MultiIndex
A weather DataFrame has been created for you.

1. Set a MultiIndex from `city` and `month`, then sort it
2. Retrieve all rows for `'London'`
3. Retrieve the single row for `('Tokyo', 'Jul')`
4. Find max temperature per city using NumPy

In [53]:
# Starter data — don't change this
np.random.seed(3)
cities = ['London', 'Tokyo', 'Sydney', 'Cairo']
months = ['Jan', 'Apr', 'Jul', 'Oct']
weather = pd.DataFrame({
    'city':   cities * 4,
    'month':  [m for m in months for _ in cities],
    'temp':   np.round(np.random.uniform(-5, 40, size=16), 1),
    'precip': np.round(np.random.uniform(0, 120, size=16), 1),
})

# Your code here
weather = weather.set_index(['city','month']).sort_index()
weather.loc['London']
weather.loc[('Tokyo','Jul')]
weather.groupby('city')['temp'].agg('max')
for city in weather.index.get_level_values('city').unique():
    print(np.max(weather.loc[city]['temp']))
weather.groupby('city')['temp'].agg(np.max)


21.6
35.2
25.4
35.3


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_49320/2729070744.py:19: FutureWarning: The provided callable <function max at 0x7febecd61af0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  weather.groupby('city')['temp'].agg(np.max)


city
Cairo     21.6
London    35.2
Sydney    25.4
Tokyo     35.3
Name: temp, dtype: float64

In [51]:
weather.index.get_level_values('city').unique()

Index(['Cairo', 'London', 'Sydney', 'Tokyo'], dtype='object', name='city')

### Task 2: Duplicates
An orders DataFrame has been created for you.

1. Count fully duplicate rows
2. Drop duplicates, keeping the **first** occurrence
3. Find which `order_id` values appear more than once
4. Show transaction counts per `status`, sorted highest first

In [13]:
# Starter data — don't change this
orders = pd.DataFrame({
    'order_id': ['O1', 'O2', 'O3', 'O2', 'O4', 'O5', 'O1', 'O3', 'O5', 'O4'],
    'status':   ['Shipped', 'Pending', 'Delivered', 'Pending', 'Shipped',
                 'Delivered', 'Shipped', 'Delivered', 'Delivered', 'Shipped'],
    'amount':   [120.0, 45.0, 300.0, 45.0, 89.0, 210.0, 120.0, 300.0, 210.0, 89.0],
    'month':    ['Jan', 'Jan', 'Feb', 'Jan', 'Mar', 'Feb', 'Jan', 'Feb', 'Feb', 'Mar'],
})

# Your code here
orders.duplicated().sum()
orders = orders.drop_duplicates(keep='first')
o = orders['order_id'].value_counts()
o[o>1].index
orders['status'].value_counts().sort_values(ascending = False)

status
Shipped      2
Delivered    2
Pending      1
Name: count, dtype: int64

---
## Level 2 — Transformations

### Task 3: stack() and unstack()
A wide DataFrame of quarterly revenue by product has been created for you
(products as the index, quarters as columns).

1. Use `.stack()` to reshape it into long format — each row one `(product, quarter)` pair. Store as `rev_long`.
2. From `rev_long`, use `.unstack()` to restore the wide shape.
3. From `rev_long`, use `.unstack(level=0)` — what do the columns represent?
4. Find the product with the highest mean revenue across all quarters using NumPy

In [21]:
# Starter data — don't change this
np.random.seed(7)
rev = pd.DataFrame({
    'product': ['Laptop', 'Phone', 'Tablet', 'Watch', 'Headphones'],
    'Q1': np.random.randint(10_000, 80_000, size=5),
    'Q2': np.random.randint(10_000, 80_000, size=5),
    'Q3': np.random.randint(10_000, 80_000, size=5),
    'Q4': np.random.randint(10_000, 80_000, size=5),
}).set_index('product')

# Your code here
rev_long = rev.stack()
rev_long
rev_long.unstack()
rev_long.unstack(level=0) ## columns represnt products
rev.mean(axis=1).idxmax()

'Headphones'

### Task 4: Custom Column Logic
An employees DataFrame has been created for you.

1. Add a `band` column: `'Senior'` if `years_exp` ≥ 8, `'Mid'` if ≥ 3, else `'Junior'`
2. Add a `bonus_eligible` column: `True` if `band` is `'Senior'` **and** `performance` ≥ 80
3. Add a `log_salary` column using NumPy
4. Count employees per band

In [25]:
# Starter data — don't change this
np.random.seed(21)
employees = pd.DataFrame({
    'name':        ['Ana', 'Ben', 'Cara', 'Dan', 'Eva', 'Finn', 'Gina', 'Hiro', 'Isla', 'Jake'],
    'years_exp':   np.random.randint(0, 15, size=10),
    'performance': np.random.randint(60, 100, size=10),
    'salary':      np.random.randint(40_000, 120_000, size=10),
    'dept':        np.random.choice(['Eng', 'Sales', 'HR'], size=10),
})

# Your code here
employees['band'] = pd.cut(employees['years_exp'], bins=[0,3,8,np.inf],
                           labels=['Junior','Mid','Senior'],
                           right=False)
employees['bonus_eligible'] = (employees['band']=='Senior') & (employees['performance']>=80)
employees['log_salary'] = np.log(employees['salary'])
employees['band'].value_counts()


band
Junior    4
Senior    4
Mid       2
Name: count, dtype: int64

---
## Level 3 — Aggregation

### Task 5: .xs() — Cross-Section
A DataFrame of monthly sales by department and region has been created for you,
with a `(department, region)` MultiIndex.

1. Use `.xs()` to retrieve all rows for `'Q2'` across all departments
2. Use `.xs()` to retrieve all rows for the `'Engineering'` department
3. Find which region had the highest total sales in Q2 using NumPy
4. Find mean sales per quarter — group by the `'quarter'` level

In [37]:
# Starter data — don't change this
np.random.seed(14)
depts    = ['Engineering', 'Marketing', 'Sales', 'Support']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']
dept_sales = pd.DataFrame({
    'dept':    depts * 4,
    'quarter': [q for q in quarters for _ in depts],
    'sales':   np.random.randint(20_000, 150_000, size=16),
    'headcount': np.random.randint(5, 50, size=16),
}).set_index(['dept', 'quarter']).sort_index()

# Your code here
dept_sales.xs('Q2',level='quarter')
dept_sales.xs('Engineering', level ='dept')
dept_sales['total'] = dept_sales['sales'] * dept_sales['headcount']
d = dept_sales.xs('Q2',level='quarter')
d['total'].idxmax()
dept_sales.groupby('quarter')['sales'].mean()

quarter
Q1    100793.25
Q2     78916.00
Q3     92716.75
Q4    110626.00
Name: sales, dtype: float64

### Task 6: Rolling Windows
A daily stock price DataFrame has been created for you.

1. Compute a 7-day rolling mean and rolling std of `price`
2. Add an `above_avg` column — `True` when `price` exceeds its 7-day rolling mean
3. Count `True` values in `above_avg`
4. Find the date where the gap between `price` and its 7-day rolling mean was largest

In [41]:
# Starter data — don't change this
np.random.seed(55)
stocks = pd.DataFrame({
    'date':  pd.date_range('2024-01-01', periods=90, freq='B'),
    'price': np.round(np.random.uniform(100, 400, size=90), 2),
}).set_index('date')

# Your code here
stocks.rolling(7).agg(['mean','std'])
stocks['rm'] = stocks.rolling(7).mean()
stocks['above_avg'] = stocks['price']>stocks['rm']
stocks['above_avg'].sum()
(stocks['price']-stocks['rm']).idxmax()

Timestamp('2024-04-04 00:00:00')

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Load the penguins dataset and run a full analysis:

1. Drop rows with any null values
2. Add a `size_ratio` column = `body_mass_g / flipper_length_mm`
3. Bin `body_mass_g` into 3 groups labelled `['Light', 'Medium', 'Heavy']`, store as `mass_group`
4. Compute mean `flipper_length_mm` grouped by `species` and `island`, then `.unstack()` so species are rows and islands are columns
5. Use `.xs()` to pull out just the `'Biscoe'` island column from the grouped result (before unstacking)
6. Find the correlation between `bill_length_mm` and `bill_depth_mm` using NumPy

In [49]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
penguins = pd.read_csv(url)

# Your code here
penguins = penguins.dropna()
penguins['size_ratio'] = penguins['body_mass_g']/penguins['flipper_length_mm']
penguins['mass_group'] = pd.cut(penguins['body_mass_g'],
                                3,
                                labels=['Light','Medium','Heavy'])
p = penguins.groupby(['species','island'])['flipper_length_mm'].mean()
p.unstack()
p.xs('Biscoe', level='island')
np.corrcoef(penguins['bill_length_mm'], penguins['bill_depth_mm'])

array([[ 1.        , -0.22862564],
       [-0.22862564,  1.        ]])